# SGD vs Adam vs AdamW — interactive comparison
Simulates 20 training steps on a single weight and plots:
- **Top**: weight value over time
- **Bottom**: actual decay applied each step (shows Adam's inconsistency)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from ipywidgets import interact, FloatSlider, Layout
%matplotlib inline

In [2]:
def grad_at(step):
    """Slightly varying gradient to make the plot interesting."""
    return 0.5 + 0.1 * np.sin(step * 0.8)


def simulate(lr=0.05, wd=0.10, w0=2.0, steps=20, b1=0.9, b2=0.999, eps=1e-8):
    """
    Run SGD / Adam (coupled decay) / AdamW (decoupled decay) for `steps` iterations.
    Returns a dict of lists: weight trajectory and decay-applied-per-step for each optimizer.
    """
    # --- SGD state ---
    sgd_w, sgd_v = w0, 0.0

    # --- Adam state ---
    adam_w, adam_m, adam_v = w0, 0.0, 0.0

    # --- AdamW state ---
    adamw_w, adamw_m, adamw_v = w0, 0.0, 0.0

    history = {
        'sgd_w': [], 'sgd_decay': [],
        'adam_w': [], 'adam_decay': [],
        'adamw_w': [], 'adamw_decay': [],
    }

    for t in range(1, steps + 1):
        g = grad_at(t)

        # ── SGD with momentum + weight decay ──────────────────────────────
        sgd_v = b1 * sgd_v + g
        sgd_decay_applied = lr * wd * sgd_w          # decay before step
        sgd_w = sgd_w - lr * sgd_v - sgd_decay_applied

        # ── Adam (coupled: decay baked into gradient) ─────────────────────
        g_mod = g + wd * adam_w                       # decay mixed INTO gradient
        adam_m = b1 * adam_m + (1 - b1) * g_mod
        adam_v = b2 * adam_v + (1 - b2) * g_mod ** 2
        m_hat  = adam_m / (1 - b1 ** t)
        v_hat  = adam_v / (1 - b2 ** t)
        adam_step = lr * m_hat / (np.sqrt(v_hat) + eps)
        # effective decay = what the wd*w term contributed after normalisation
        # approximate: compare step with pure-g step
        adam_m_pure = b1 * 0 + (1 - b1) * g   # what m would be with pure g (rough)
        adam_v_pure = b2 * 0 + (1 - b2) * g ** 2
        m_hat_pure  = adam_m_pure / (1 - b1 ** t)
        v_hat_pure  = adam_v_pure / (1 - b2 ** t)
        pure_step   = lr * m_hat_pure / (np.sqrt(v_hat_pure) + eps)
        adam_decay_applied = abs(adam_step - pure_step)   # how much decay actually added
        adam_w = adam_w - adam_step

        # ── AdamW (decoupled: decay applied separately after gradient step) 
        adamw_m = b1 * adamw_m + (1 - b1) * g        # pure gradient only
        adamw_v = b2 * adamw_v + (1 - b2) * g ** 2
        wm_hat  = adamw_m / (1 - b1 ** t)
        wv_hat  = adamw_v / (1 - b2 ** t)
        adamw_grad_step    = lr * wm_hat / (np.sqrt(wv_hat) + eps)
        adamw_decay_applied = lr * wd * adamw_w       # clean, predictable
        adamw_w = adamw_w - adamw_grad_step - adamw_decay_applied

        history['sgd_w'].append(sgd_w)
        history['sgd_decay'].append(sgd_decay_applied)
        history['adam_w'].append(adam_w)
        history['adam_decay'].append(adam_decay_applied)
        history['adamw_w'].append(adamw_w)
        history['adamw_decay'].append(adamw_decay_applied)

    return history

In [ ]:
def plot_optimizers(lr=0.05, weight_decay=0.10, initial_weight=2.0):
    h = simulate(lr=lr, wd=weight_decay, w0=initial_weight)
    steps = np.arange(1, 21)

    C_SGD   = '#185FA5'
    C_ADAM  = '#D85A30'
    C_ADAMW = '#1D9E75'

    fig = plt.figure(figsize=(11, 7))
    gs  = gridspec.GridSpec(2, 1, hspace=0.45)

    # ── top: weight trajectory ────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(steps, h['sgd_w'],   color=C_SGD,   lw=2, marker='o', ms=4, label='SGD')
    ax1.plot(steps, h['adam_w'],  color=C_ADAM,  lw=2, marker='o', ms=4, label='Adam (coupled)')
    ax1.plot(steps, h['adamw_w'], color=C_ADAMW, lw=2, marker='o', ms=4, label='AdamW (decoupled)')
    ax1.set_title('Weight value over training steps', fontsize=13, pad=8)
    ax1.set_xlabel('step')
    ax1.set_ylabel('w')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.25)
    ax1.set_xticks(steps)

    # ── bottom: decay actually applied each step ──────────────────────────
    ax2 = fig.add_subplot(gs[1])
    ax2.plot(steps, h['sgd_decay'],   color=C_SGD,   lw=2, marker='s', ms=4,
             linestyle='--', label='SGD decay')
    ax2.plot(steps, h['adam_decay'],  color=C_ADAM,  lw=2, marker='s', ms=4,
             linestyle='--', label='Adam decay (distorted)')
    ax2.plot(steps, h['adamw_decay'], color=C_ADAMW, lw=2, marker='s', ms=4,
             linestyle='--', label='AdamW decay (clean)')
    ax2.set_title('Actual decay applied per step', fontsize=13, pad=8)
    ax2.set_xlabel('step')
    ax2.set_ylabel('decay amount')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.25)
    ax2.set_xticks(steps)

    fig.suptitle(
        f'lr={lr:.3f}  |  λ={weight_decay:.2f}  |  w₀={initial_weight:.1f}',
        fontsize=11, y=1.01, color='gray'
    )
    plt.show()


slider_layout = Layout(width='400px')

interact(
    plot_optimizers,
    lr            = FloatSlider(value=0.05, min=0.01, max=0.20, step=0.01,
                                description='learning rate', layout=slider_layout,
                                style={'description_width': '120px'}),
    weight_decay  = FloatSlider(value=0.10, min=0.00, max=0.30, step=0.01,
                                description='weight decay λ', layout=slider_layout,
                                style={'description_width': '120px'}),
    initial_weight= FloatSlider(value=2.0,  min=0.5,  max=5.0,  step=0.1,
                                description='initial weight', layout=slider_layout,
                                style={'description_width': '120px'}),
);

interactive(children=(FloatSlider(value=0.05, description='learning rate', layout=Layout(width='400px'), max=0…